<a href="https://colab.research.google.com/github/FABYG6/Machine_Learning/blob/main/Informe_de_Trabajo_ML_Supervisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# CASO DE ESTUDIO: RECURSOS HUMANOS
# Predicción de rotación de empleados
# ============================================================

import pandas as pd

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Procesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Evaluación
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. DATASET
# ============================================================

# renuncia:
# 0 = permanece
# 1 = renuncia

data = {
    "edad": [
        25, 28, 35, 40, 30, 45, 32, 29, 50, 38,
        27, 33, 41, 36, 31, 48, 26, 39, 42, 34
    ],
    "salario": [
        1800, 2000, 3500, 4500, 2200, 5000, 2800, 2100, 6000, 4000,
        1900, 3000, 4700, 3600, 2500, 5500, 1700, 4200, 4800, 3200
    ],
    "años_empresa": [
        1, 2, 5, 8, 2, 10, 4, 2, 15, 7,
        1, 4, 9, 6, 3, 12, 1, 8, 10, 5
    ],
    "satisfaccion": [
        2, 3, 7, 8, 3, 9, 5, 2, 9, 7,
        2, 6, 8, 6, 4, 9, 1, 7, 8, 5
    ],
    "horas_extra": [
        15, 12, 5, 4, 14, 3, 8, 13, 2, 6,
        16, 7, 4, 6, 10, 3, 18, 5, 4, 8
    ],
    "renuncia": [
        1, 1, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0
    ]
}

df = pd.DataFrame(data)

print("Dataset inicial:")
print(df.head())

# ============================================================
# 2. VARIABLES X e y
# ============================================================

X = df[
    [
        "edad",
        "salario",
        "años_empresa",
        "satisfaccion",
        "horas_extra"
    ]
]

y = df["renuncia"]

# ============================================================
# 3. DIVISIÓN DE DATOS
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# ============================================================
# 4. MODELOS
# ============================================================

modelos = {
    "Regresión Logística": make_pipeline (StandardScaler(), LogisticRegression(max_iter=1000)),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": make_pipeline (StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "SVM": make_pipeline (StandardScaler(), SVC(kernel="linear")),
    "Naive Bayes": GaussianNB()
}

# ============================================================
# 5. EVALUACIÓN DE MODELOS
# ============================================================

resultados = []

for nombre, modelo in modelos.items():

    print("\n" + "="*70)
    print("MODELO:", nombre)
    print("="*70)

    # Entrenamiento
    modelo.fit(X_train, y_train)

    # Predicción
    y_pred = modelo.predict(X_test)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Validación cruzada
    cv_scores = cross_val_score(
        modelo,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    cv_promedio = cv_scores.mean()

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Validación cruzada": cv_promedio
    })

    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))

    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("Validación cruzada:", cv_scores)
    print("Promedio:", cv_promedio)

# ============================================================
# 6. TABLA FINAL
# ============================================================

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by="F1-score", ascending=False)

print("\n" + "="*70)
print("TABLA COMPARATIVA")
print("="*70)

print(df_resultados.round(3))

# ============================================================
# 7. MEJOR MODELO
# ============================================================

mejor_modelo = df_resultados.iloc[0]

print("\n" + "=" * 70)
print("MEJOR MODELO SEGÚN F1-SCORE")
print("=" * 70)

print("Modelo:", mejor_modelo["Modelo"])
print("Accuracy:", round(mejor_modelo["Accuracy"], 3))
print("Precision:", round(mejor_modelo["Precision"], 3))
print("Recall:", round(mejor_modelo["Recall"], 3))
print("F1-score:", round(mejor_modelo["F1-score"], 3))
print("Validación cruzada:",round(mejor_modelo["Validación cruzada"], 3)
)

Dataset inicial:
   edad  salario  años_empresa  satisfaccion  horas_extra  renuncia
0    25     1800             1             2           15         1
1    28     2000             2             3           12         1
2    35     3500             5             7            5         0
3    40     4500             8             8            4         0
4    30     2200             2             3           14         1

MODELO: Regresión Logística

Matriz de confusión:
[[4 0]
 [0 2]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Validación cruzada: [1. 1. 1. 1. 1.]
Promedio: 1.0

MODELO: Árbol de Decisión

Matriz de confusión:
[[4 0]
 [0 2]]

Reporte de clasificación:
             